# Interactive Charts with Plotly & Altair

Interactive charts let readers explore: hover for details, zoom into a period, filter by a selection, play an animation. **Plotly** produces rich interactive charts from a few lines of code; **Altair** is a declarative *grammar of graphics* for Python (based on Vega-Lite) with powerful linked selections.

**After this lesson you can:**

1. Build interactive scatter, line and animated charts with Plotly Express.
2. Customise layout, hover text and range controls.
3. Describe charts declaratively with Altair encodings.
4. Link several Altair charts with selections (brushing and filtering).
5. Export interactive charts as HTML.

**Data:** Gapminder, NYC taxi trips (aggregated) and Palmer penguins.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads pinned public datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

import altair as alt
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"      # small output: loads plotly.js from a CDN
gap = pd.read_json(datasets.fetch("gapminder.json"))
REGIONS = {0: "South Asia", 1: "Europe & Central Asia", 2: "Sub-Saharan Africa", 3: "Americas", 4: "East Asia & Pacific",
           5: "Middle East & North Africa"}
gap["region"] = gap["cluster"].map(REGIONS)
import plotly
print("plotly", plotly.__version__, "| altair", alt.__version__)

plotly 5.24.1 | altair 6.3.0


## 1. Plotly Express: an Animated Scatter Plot

Hans Rosling's famous chart: fertility vs life expectancy, bubble size = population, one frame per year. Press play, hover over a bubble, or click a region in the legend to hide it.

In [2]:
fig = px.scatter(gap, x="fertility", y="life_expect", size="pop", color="region", hover_name="country",
                 animation_frame="year", size_max=45, range_x=[0.5, 9], range_y=[25, 90],
                 labels={"fertility": "Children per woman", "life_expect": "Life expectancy (years)"},
                 title="Fewer children, longer lives: 1955-2005")
fig.update_layout(height=520, legend_title_text="")
fig.show()

## 2. Time Series with a Range Slider

A range slider and range buttons let the reader zoom into a period without writing code.

In [3]:
taxi = pd.read_parquet(datasets.taxi("2024-01"), columns=["tpep_pickup_datetime", "total_amount"])
taxi = taxi[taxi["tpep_pickup_datetime"].between("2024-01-01", "2024-02-01", inclusive="left")]
hourly = taxi.set_index("tpep_pickup_datetime").resample("h").agg(trips=("total_amount", "size"), revenue=("total_amount", "sum")).reset_index()

fig = px.line(hourly, x="tpep_pickup_datetime", y="trips", title="Yellow taxi trips per hour, January 2024",
              labels={"tpep_pickup_datetime": "", "trips": "Trips per hour"})
fig.update_traces(hovertemplate="%{x|%a %d %b, %H:00}<br>%{y:,} trips<extra></extra>")
fig.update_xaxes(rangeslider_visible=True, rangeselector=dict(buttons=[
    dict(count=1, label="1 day", step="day", stepmode="backward"),
    dict(count=7, label="1 week", step="day", stepmode="backward"),
    dict(step="all", label="all")]))
fig.update_layout(height=450)
fig.show()

## 3. Altair: a Grammar of Graphics

In Altair, a chart is a **description**: data + mark + encodings of fields to channels (x, y, colour, size, tooltip). Field types are declared with a suffix: `:Q` quantitative, `:N` nominal, `:O` ordinal, `:T` temporal. Altair sends the data to the browser, so aggregate or sample large data first (the default limit is 5,000 rows).

In [4]:
penguins = pd.read_json(datasets.fetch("penguins.json")).rename(columns=lambda c: c.split(" (")[0].lower().replace(" ", "_")).dropna()
base = alt.Chart(penguins).mark_circle(size=40, opacity=0.7).encode(
    x=alt.X("flipper_length:Q", scale=alt.Scale(zero=False), title="Flipper length (mm)"),
    y=alt.Y("body_mass:Q", scale=alt.Scale(zero=False), title="Body mass (g)"),
    color=alt.Color("species:N", scale=alt.Scale(range=["#E69F00", "#56B4E9", "#009E73"])),
    tooltip=["species", "island", "sex", "body_mass"],
)
trend = base.transform_regression("flipper_length", "body_mass", groupby=["species"]).mark_line(size=2)
(base + trend).properties(width=480, height=320, title="Layered chart: points + regression lines per species")

alt.LayerChart(...)

## 4. Linked Selections: Brushing and Filtering

Draw a rectangle on the scatter plot (**brush**): the histogram on the right shows only the selected penguins. Click a species in the legend to filter both charts. This **linked-view** pattern is the core of interactive dashboards (Lesson 11).

In [5]:
brush = alt.selection_interval(encodings=["x", "y"])
legend_pick = alt.selection_point(fields=["species"], bind="legend")
scatter = alt.Chart(penguins).mark_circle(size=45).encode(
    x=alt.X("beak_length:Q", scale=alt.Scale(zero=False), title="Beak length (mm)"),
    y=alt.Y("beak_depth:Q", scale=alt.Scale(zero=False), title="Beak depth (mm)"),
    color=alt.condition(brush, alt.Color("species:N", scale=alt.Scale(range=["#E69F00", "#56B4E9", "#009E73"])), alt.value("#DDDDDD")),
    opacity=alt.condition(legend_pick, alt.value(0.9), alt.value(0.1)),
).add_params(brush, legend_pick).properties(width=360, height=300, title="Brush to select")
hist = alt.Chart(penguins).mark_bar().encode(
    x=alt.X("body_mass:Q", bin=alt.Bin(maxbins=25), title="Body mass (g)"),
    y=alt.Y("count()", title="Penguins"),
    color=alt.Color("species:N", scale=alt.Scale(range=["#E69F00", "#56B4E9", "#009E73"])),
).transform_filter(brush).transform_filter(legend_pick).properties(width=360, height=300, title="Mass of the selected penguins")
scatter | hist

alt.HConcatChart(...)

## 5. Export and Sizes

Both libraries export self-contained HTML files that work in any browser and can be shared or embedded. The size depends on the data embedded in the file and on whether the JavaScript library is included or loaded from a CDN.

In [6]:
import tempfile

out = Path(tempfile.mkdtemp())
anim = px.scatter(gap, x="fertility", y="life_expect", size="pop", color="region", animation_frame="year")
anim.write_html(out / "plotly_full.html", include_plotlyjs=True)
anim.write_html(out / "plotly_cdn.html", include_plotlyjs="cdn")
(scatter | hist).save(out / "altair_linked.html")
sizes = {p.name: p.stat().st_size / 1024 for p in sorted(out.glob("*.html"))}
print({k: f"{v:,.0f} KiB" for k, v in sizes.items()})

{'altair_linked.html': '51 KiB', 'plotly_cdn.html': '59 KiB', 'plotly_full.html': '4,511 KiB'}


| | Plotly | Altair |
| :--- | :--- | :--- |
| style | figure objects, many chart types | declarative grammar (Vega-Lite) |
| strengths | 3-D, maps, animations, Dash apps | linked selections, concise layered charts |
| large data | WebGL traces (`scattergl`) handle ~10⁵–10⁶ points | aggregate first; VegaFusion for larger data |
| static export | `kaleido` (needs a headless browser) | `vl-convert` |

*Insight: Add interactivity when the reader needs to explore (details on hover, zoom, filtering), not as decoration. Keep the default view meaningful on its own, because many readers never click.*

## Real-World Use Case or Analogy

A static chart is a **printed map**; an interactive chart is a **navigation app**: you can zoom into your street, tap a place for details, and filter for petrol stations. Both are useful, and the app is useless if the first screen is confusing.

Newsrooms publish interactive charts for elections and economic data, product teams embed Plotly charts in internal tools, and data apps (Lesson 09) combine interactive charts with controls.